[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/01_unicode_bytes/01_unicode_bytes.ipynb)

# Módulo 01 · Texto como bytes: Unicode, UTF-8, normalización
**Tokenización y Embeddings: de BPE a los modelos modernos**

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración** | ≈ 4 h (lección ≈ 2.5 h + proyecto ≈ 1.5 h) |
| **GPU / coste** | Ninguna: CPU (0 unidades de Colab Pro) |
| **Prerrequisitos** | Módulo 00 (corpus Políglota, métricas, BM25). Python sólido |

> **Estado del arte de referencia: octubre de 2026.** Las cifras sujetas a cambio llevan fecha; las técnicas se etiquetan 🟢 / 🟡 / 🔴.

## Objetivos de aprendizaje
Al terminar este módulo podrás:

1. **Distinguir** y **contar** grafemas, code points y bytes de un texto, y explicar por qué `len()` significa cosas distintas en Python, JavaScript y Go/Rust.
2. **Implementar** un codificador y decodificador de **UTF-8** a mano (con validación estricta) y **verificarlo** contra el de Python en todo el espacio Unicode y con *fuzzing*.
3. **Explicar** UTF-16/UTF-32, pares sustitutos y el almacenamiento de `str` en CPython.
4. **Aplicar** NFC/NFD/NFKC/NFKD, **implementar** NFD a mano y **medir** cómo la forma de normalización cambia bytes, tokens y resultados de búsqueda.
5. **Detectar** homoglifos, caracteres invisibles y controles bidireccionales, y **explicar** el riesgo de seguridad.
6. **Cuantificar** los bytes por carácter en 12 idiomas y **relacionarlo** con el costo posterior de tokenizar (analogías con compresión y con tipología lingüística).
7. **Diagnosticar** y **reparar** *mojibake*, y elegir el manejador de errores de codificación correcto.

> **🧭 Ruta núcleo vs ampliación.** La lección tiene ≈ 74 celdas. **Ruta núcleo (≈ 2 h):** secciones 1, 2, 3 (UTF-8), 6 (normalización: tabla, impacto en tokens y en búsqueda), 8 (bytes por idioma) y 10 (mojibake). **Ampliación (≈ 0.5 h extra):** 4 (UTF-16 y `str` de CPython), 5 (grafemas), 7 (homoglifos y seguridad), 9 (compresión) y 11 (escrituras del corpus), además de la verificación exhaustiva de NFD. El proyecto asume la ruta núcleo más la sección 7.

## 🔁 Conexión con módulos anteriores
Antes de empezar, responde de memoria (despliega para comprobar):

**1.** ¿Qué diferencia hay entre el «token» de `tokenize_simple` (módulo 00) y el token de un LLM?

<details>
<summary>Ver respuesta</summary>

`tokenize_simple` produce *términos léxicos* (palabras minúsculas) para un índice invertido; un token de LLM es una unidad de un vocabulario fijo (a menudo sub-palabra o secuencia de bytes) con un id entero. El primero es parte del *retrieval léxico*; el segundo, del *modelo*.

</details>

**2.** En la frase de GPT-2 del módulo 00, ¿por qué el español costaba más tokens que el inglés? Da dos hipótesis.

<details>
<summary>Ver respuesta</summary>

(1) El vocabulario se aprendió con mayoría de inglés, así que hay menos fusiones para español. (2) «ñ», «ú», «é» ocupan 2 bytes y, sin un token para ellas, se parten en bytes. Este módulo cuantifica la hipótesis 2.

</details>

**3.** ¿Qué hace `evaluate()` con las consultas sin documentos relevantes, y por qué?

<details>
<summary>Ver respuesta</summary>

Las excluye del promedio (convención `trec_eval`): su nDCG sería 0/0. Devolver 0 las hundiría artificialmente.

</details>

**4.** ¿Qué contrato de datos devuelve `carga_corpus()`?

<details>
<summary>Ver respuesta</summary>

Un dict con `docs` (`id`, `lang`, `text`), `queries` (`qid`, `lang`, `text`), `qrels` (`qid → {doc_id: 1}`) y `source` (`real`/`synthetic`).

</details>

## 1. Intuición primero: el texto tiene tres capas

Cuando escribes «👨‍👩‍👧‍👦» ves **un** símbolo. Python dirá `len(...) == 7`. El disco guarda **25** bytes. Las tres cifras son correctas porque hablan de capas distintas:

| Capa | ¿Qué es? | Quién la define | Ejemplo «é» | Ejemplo 👨‍👩‍👧‍👦 |
|---|---|---|---|---|
| **Grafema** (*extended grapheme cluster*) | lo que el lector percibe como «un carácter» | UAX #29 | 1 | 1 |
| **Code point** | un número del catálogo Unicode (U+0000…U+10FFFF) | el estándar Unicode | 1 (U+00E9) o 2 (`e` + U+0301) | 7 |
| **Byte** | un octeto en memoria/disco, según una *codificación* | UTF-8 (RFC 3629) | 2 (`C3 A9`) | 25 |

Un tokenizer de LLM actúa sobre **bytes** (byte-level BPE, módulo 02) o sobre **code points** (SentencePiece, módulo 03); ninguno ve grafemas. Si el texto llega con una representación inesperada (é descompuesta, un carácter invisible, un homoglifo), **el modelo ve otra cosa**.

> ### 🎨 GEMINI-IMAGEN · ID: `M01-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Las tres capas del texto: grafema, code points, bytes**
>
> **Qué muestra:** Un único emoji de familia descompuesto en sus tres capas: 1 grafema, 7 code points (con 3 ZWJ) y 25 bytes UTF-8. Fija la diferencia entre lo que ves, lo que dice Unicode y lo que guarda el disco.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Emoji de familia descompuesto: 1 grafema, 7 code points y 25 bytes UTF-8
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible, sin ilustraciones decorativas.
> Composición vertical en tres franjas horizontales apiladas, unidas por flechas verticales grises que bajan de una franja a la siguiente, con la etiqueta «se descompone en» sobre cada flecha.
> Franja 1 (borde azul), título «Grafema (lo que ves): 1». Contenido: el emoji de familia «👨‍👩‍👧‍👦» en tamaño grande, centrado.
> Franja 2 (borde naranja), título «Code points (Unicode): 7». Contenido: siete cajas en fila con estos textos EXACTOS, de izquierda a derecha: «U+1F468 👨», «U+200D ZWJ», «U+1F469 👩», «U+200D ZWJ», «U+1F467 👧», «U+200D ZWJ», «U+1F466 👦». Las tres cajas ZWJ en gris con la nota «unión de ancho cero».
> Franja 3 (borde verde), título «Bytes UTF-8: 25». Contenido: siete grupos de bytes en hexadecimal bajo cada code point, EXACTOS y en este orden: «F0 9F 91 A8», «E2 80 8D», «F0 9F 91 A9», «E2 80 8D», «F0 9F 91 A7», «E2 80 8D», «F0 9F 91 A6». Bajo cada grupo, el número de bytes: «4», «3», «4», «3», «4», «3», «4».
> Título superior: «Un símbolo, tres capas». Nota inferior en gris: «len() de Python cuenta la capa 2; el disco guarda la capa 3».
> Evita: cambiar los números o los códigos, texto ilegible, iconos decorativos, flechas ambiguas, más de tres franjas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M01-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Emoji de familia descompuesto: 1 grafema, 7 code points y 25 bytes UTF-8](assets/M01-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Grafema (1):        👨‍👩‍👧‍👦
>                       │ se descompone en
> Code points (7):    U+1F468  U+200D  U+1F469  U+200D  U+1F467  U+200D  U+1F466
>                       👨      ZWJ      👩      ZWJ      👧      ZWJ      👦
>                       │ se codifica en UTF-8
> Bytes (25):        F0 9F 91 A8|E2 80 8D|F0 9F 91 A9|E2 80 8D|F0 9F 91 A7|E2 80 8D|F0 9F 91 A6
>                        4          3          4          3          4          3          4
> ```

> 👀 **Qué debes observar.** Cuenta de izquierda a derecha: 4+3+4+3+4+3+4 = **25** bytes. Los tres `ZWJ` (U+200D) **no se ven** pero ocupan 3 bytes cada uno y son code points de pleno derecho: un tokenizer byte-level tiene que pagar por ellos. En este emoji, 9 de los 25 bytes (3 ZWJ × 3 bytes) son «pegamento» invisible y los otros 16 son los cuatro emojis (4 × 4 bytes).

## 2. Code points: Unicode como catálogo

Unicode asigna a cada carácter abstracto un **code point**: un entero entre $0$ y $10FFFF_{16}$ ($1\,114\,112$ valores, repartidos en 17 *planos* de 65 536). Se escribe `U+0041` (letra A). Cada code point tiene **nombre**, **categoría general** (L=letra, M=marca, N=número, P=puntuación, S=símbolo, Z=separador, C=otros) y propiedades (escritura, ancho, comportamiento bidireccional…). Unicode **no** dice cómo se guarda en bytes: eso lo hacen las *codificaciones* (UTF-8, UTF-16, UTF-32).

Los code points `U+D800…U+DFFF` (2 048) son *surrogates*, reservados para UTF-16 y **no** son caracteres válidos: los **valores escalares** Unicode son los otros $1\,112\,064$.

In [ ]:
!pip install -q numpy pandas matplotlib regex tiktoken rank_bm25 ftfy

In [ ]:
import matplotlib.pyplot as plt

AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11})

import random, sys, time, unicodedata, collections, math, re
import numpy as np, pandas as pd, regex

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True
print(f"python {sys.version.split()[0]} | Unicode (unicodedata) {unicodedata.unidata_version} | regex {regex.__version__}")

In [ ]:
def inspecciona(s: str) -> pd.DataFrame:
    filas = []
    for ch in s:
        filas.append({"carácter": ch, "code point": f"U+{ord(ch):04X}", "nombre": unicodedata.name(ch, "(sin nombre)"),
                      "categoría": unicodedata.category(ch), "bytes UTF-8": ch.encode("utf-8").hex(" ").upper()})
    return pd.DataFrame(filas)

display(inspecciona("Añ€😀한"))
asignados = sum(unicodedata.category(chr(c)) not in ("Cn", "Co", "Cs") for c in range(0x110000))
print(f"Code points con carácter asignado (sin privados ni surrogates) en Unicode {unicodedata.unidata_version}: {asignados:,}")

## 3. UTF-8 desde cero

### Diseño
UTF-8 (Pike & Thompson, 1992; RFC 3629) codifica cada code point en **1 a 4 bytes**. Es un **código de prefijo de longitud variable** (como los de Huffman, pero con una tabla fija y pensada para ASCII): el **primer byte** dice cuántos bytes tiene la secuencia, los demás son *bytes de continuación* `10xxxxxx`.

| Rango de code points | Bytes | Patrón binario | Bits útiles |
|---|---|---|---|
| `U+0000…U+007F` | 1 | `0xxxxxxx` | 7 |
| `U+0080…U+07FF` | 2 | `110xxxxx 10xxxxxx` | 11 |
| `U+0800…U+FFFF` | 3 | `1110xxxx 10xxxxxx 10xxxxxx` | 16 |
| `U+10000…U+10FFFF` | 4 | `11110xxx 10xxxxxx 10xxxxxx 10xxxxxx` | 21 |

**Ejemplo a mano:** `€` = `U+20AC` = `0010 0000 1010 1100` (16 bits). Rango de 3 bytes: parte los 16 bits en 4 + 6 + 6 → `0010 | 000010 | 101100` → `11100010 | 10000010 | 10101100` = **`E2 82 AC`**.

**Propiedades que lo hicieron ganar:**
* **Compatible con ASCII**: un archivo ASCII *ya es* UTF-8 válido.
* **Autosincronizante**: un byte `10xxxxxx` nunca puede iniciar un carácter, así que si pierdes bytes, resincronizas en el siguiente byte que no sea de continuación.
* **Prefix-free**: ninguna secuencia válida es prefijo de otra → se decodifica sin ambigüedad.
* **Ordenación**: comparar bytes equivale a comparar code points.
* **Sin *overlongs***: cada code point tiene **una sola** codificación válida (la más corta); `C0 80` para U+0000 está *prohibido*: aceptarlo fue una fuente histórica de fallos de seguridad al saltarse filtros.

In [ ]:
def utf8_encode_cp(cp: int) -> bytes:
    """Codifica UN valor escalar Unicode a bytes UTF-8 (a mano, con máscaras de bits)."""
    if not (0 <= cp <= 0x10FFFF) or 0xD800 <= cp <= 0xDFFF:
        raise ValueError(f"U+{cp:X} no es un valor escalar Unicode")
    if cp < 0x80:
        return bytes([cp])
    if cp < 0x800:
        return bytes([0xC0 | cp >> 6, 0x80 | cp & 0x3F])
    if cp < 0x10000:
        return bytes([0xE0 | cp >> 12, 0x80 | (cp >> 6) & 0x3F, 0x80 | cp & 0x3F])
    return bytes([0xF0 | cp >> 18, 0x80 | (cp >> 12) & 0x3F, 0x80 | (cp >> 6) & 0x3F, 0x80 | cp & 0x3F])

def utf8_encode(s: str) -> bytes:
    return b"".join(utf8_encode_cp(ord(c)) for c in s)

def utf8_decode(b: bytes) -> str:
    """Decodificador ESTRICTO: rechaza bytes iniciales inválidos, continuaciones erróneas, truncados, overlongs y surrogates."""
    out, i, n = [], 0, len(b)
    while i < n:
        b0 = b[i]
        if b0 < 0x80:                 need, cp, minimo = 0, b0, 0
        elif 0xC2 <= b0 <= 0xDF:      need, cp, minimo = 1, b0 & 0x1F, 0x80      # C0 y C1 solo darían overlongs
        elif 0xE0 <= b0 <= 0xEF:      need, cp, minimo = 2, b0 & 0x0F, 0x800
        elif 0xF0 <= b0 <= 0xF4:      need, cp, minimo = 3, b0 & 0x07, 0x10000   # F5..FF superan U+10FFFF
        else:
            raise ValueError(f"byte inicial inválido 0x{b0:02X} en la posición {i}")
        for k in range(1, need + 1):
            if i + k >= n or (b[i + k] & 0xC0) != 0x80:
                raise ValueError(f"continuación inválida o secuencia truncada en {i + k}")
            cp = cp << 6 | (b[i + k] & 0x3F)
        if cp < minimo:
            raise ValueError(f"codificación demasiado larga (overlong) en {i}")
        if 0xD800 <= cp <= 0xDFFF or cp > 0x10FFFF:
            raise ValueError(f"valor escalar inválido U+{cp:X} en {i}")
        out.append(chr(cp)); i += need + 1
    return "".join(out)

print("€ →", utf8_encode("€").hex(" ").upper(), "| 😀 →", utf8_encode("😀").hex(" ").upper(), "| ñ →", utf8_encode("ñ").hex(" ").upper())
assert utf8_decode(utf8_encode("Añ€😀한")) == "Añ€😀한"

### Verificación contra Python
Una implementación «parece correcta» con 5 ejemplos; la **verificamos** de dos formas: (1) **exhaustiva** sobre los 1 112 064 valores escalares (codificar y decodificar), y (2) **fuzzing**: 50 000 secuencias de bytes aleatorias (con bytes elegidos para provocar overlongs, surrogates, truncados y continuaciones sueltas) donde mi decodificador debe aceptar **exactamente** lo mismo que el de Python y producir el mismo texto.

In [ ]:
t0 = time.time()
todos = [chr(c) for c in range(0x110000) if not 0xD800 <= c <= 0xDFFF]
mal = sum(utf8_encode_cp(ord(c)) != c.encode("utf-8") for c in todos)
texto = "".join(todos)
assert utf8_decode(texto.encode("utf-8")) == texto
print(f"exhaustivo: {len(todos):,} valores escalares, discrepancias de codificación = {mal}  ({time.time() - t0:.1f} s)")

rng = random.Random(SEED); discrep = 0; casos = collections.Counter()
alfabeto = [0x41, 0xC0, 0xC1, 0xC2, 0xC3, 0xE0, 0xE2, 0xED, 0xF0, 0xF4, 0xF5, 0x80, 0xA9, 0xBF, 0x9F, 0x8F]
for _ in range(50_000):
    b = bytes(rng.choice(alfabeto) for _ in range(rng.randint(1, 5)))
    try: mio = utf8_decode(b)
    except ValueError: mio = None
    try: ref = b.decode("utf-8")
    except UnicodeDecodeError: ref = None
    casos["válido" if ref is not None else "inválido"] += 1
    discrep += mio != ref
print(f"fuzzing: {sum(casos.values()):,} secuencias ({dict(casos)}), discrepancias = {discrep}")
assert mal == 0 and discrep == 0

In [ ]:
# Rangos de UTF-8: cuántos code points caben en 1, 2, 3 y 4 bytes (y cuántos son válidos)
rangos = [("1 byte\n(U+0000–U+007F)", 0x80, 7), ("2 bytes\n(U+0080–U+07FF)", 0x800 - 0x80, 11),
          ("3 bytes\n(U+0800–U+FFFF)", 0x10000 - 0x800 - 2048, 16), ("4 bytes\n(U+10000–U+10FFFF)", 0x110000 - 0x10000, 21)]
fig, ax = plt.subplots(figsize=(8, 3.8))
b = ax.bar([r[0] for r in rangos], [r[1] for r in rangos], color=[VERDE, AZUL, NARANJA, GRIS])
ax.set_yscale("log"); ax.set_ylabel("valores escalares (escala log)"); ax.set_title("Capacidad de cada longitud de UTF-8")
for rect, (_, n, bits) in zip(b, rangos):
    ax.text(rect.get_x() + rect.get_width() / 2, n * 1.2, f"{n:,}\n{bits} bits útiles", ha="center", fontsize=8)
ax.set_ylim(50, 4e6); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con la escala logarítmica ves lo asimétrico del diseño: **128** valores en 1 byte (ASCII), pero más de **un millón** en 4 bytes. Un texto en inglés casi no usa los rangos largos; un texto en chino vive en los 3 bytes. Esa asimetría es, literalmente, el «impuesto de bytes» de muchos idiomas.

> ### 🎨 GEMINI-IMAGEN · ID: `M01-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Estructura de bits de UTF-8 con un ejemplo resuelto**
>
> **Qué muestra:** La tabla de patrones de bits de UTF-8 y la codificación paso a paso del símbolo € (U+20AC → E2 82 AC), para consultarla mientras implementas el codificador.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tabla de patrones de bits de UTF-8 y codificación del símbolo euro paso a paso
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB para bits de payload, naranja #F28E2B para bits de encabezado (prefijos), verde #2CA02C para el resultado, gris #6B7280 para anotaciones. Tipografía monoespaciada para los bits y sans-serif para títulos, sin ilustraciones decorativas.
> Mitad superior: una tabla de 4 filas, cada una con una celda por byte, con los prefijos en naranja y los bits «x» en azul. Textos EXACTOS por fila:
> Fila 1: «1 byte · U+0000–U+007F» y «0xxxxxxx» (7 bits útiles).
> Fila 2: «2 bytes · U+0080–U+07FF» y «110xxxxx | 10xxxxxx» (11 bits útiles).
> Fila 3: «3 bytes · U+0800–U+FFFF» y «1110xxxx | 10xxxxxx | 10xxxxxx» (16 bits útiles).
> Fila 4: «4 bytes · U+10000–U+10FFFF» y «11110xxx | 10xxxxxx | 10xxxxxx | 10xxxxxx» (21 bits útiles).
> Mitad inferior, título «Ejemplo: € = U+20AC»: una cadena de tres pasos de izquierda a derecha con flechas:
> Paso 1: «U+20AC = 0010 0000 1010 1100 (16 bits)».
> Paso 2: «Reparte en 4 + 6 + 6 bits: 0010 | 000010 | 101100».
> Paso 3: «Añade prefijos: 11100010 | 10000010 | 10101100 = E2 82 AC», con los prefijos «1110», «10» y «10» en naranja y el resultado en un recuadro verde.
> Título superior: «Cómo se codifica un code point en UTF-8».
> Evita: cambiar bits o rangos, texto ilegible, elementos decorativos, mezclar los colores de prefijo y payload.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M01-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Tabla de patrones de bits de UTF-8 y codificación del símbolo euro paso a paso](assets/M01-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> 1 byte  U+0000–U+007F      0xxxxxxx
> 2 bytes U+0080–U+07FF      110xxxxx 10xxxxxx
> 3 bytes U+0800–U+FFFF      1110xxxx 10xxxxxx 10xxxxxx
> 4 bytes U+10000–U+10FFFF   11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
> 
> € = U+20AC = 0010 0000 1010 1100
>   → 0010 | 000010 | 101100
>   → 11100010 | 10000010 | 10101100 = E2 82 AC
> ```

> 👀 **Qué debes observar.** Comprueba que los 3 bytes de `€` coinciden con la salida de `utf8_encode("€")` (E2 82 AC). Fíjate en que los prefijos (`1110`, `10`, `10`) **nunca** aparecen como payload: por eso un byte `10xxxxxx` suelto delata que estás a mitad de un carácter.

In [ ]:
# Autosincronización: arranca la lectura en mitad de una secuencia y resincroniza en el siguiente byte que no sea de continuación
def resincroniza(b: bytes) -> str:
    i = 0
    while i < len(b) and (b[i] & 0xC0) == 0x80:      # descarta continuaciones huérfanas
        i += 1
    return utf8_decode(b[i:])

datos = "señal 😀 niño".encode("utf-8")
print("completo        :", datos.decode("utf-8"))
print("cortado en 'ñ'  :", datos[3:].decode("utf-8", errors="replace"), "(Python reemplaza con U+FFFD)")
print("resincronizado  :", resincroniza(datos[3:]))
# Prefijo único: el primer byte determina la longitud (código de prefijo)
longitud = lambda b0: 1 if b0 < 0x80 else 2 if 0xC2 <= b0 <= 0xDF else 3 if 0xE0 <= b0 <= 0xEF else 4 if 0xF0 <= b0 <= 0xF4 else 0
print("longitudes por byte inicial:", {hex(b): longitud(b) for b in (0x41, 0xC3, 0xE2, 0xF0, 0x80, 0xC0)})

> ### 🧪 Mini-experimento: los *overlongs* y por qué un decodificador estricto importa
> `C0 80` sería una forma «larga» de escribir `U+0000`. Un decodificador permisivo la aceptaría, y un filtro de seguridad que busque el byte `00` (o `/`, que se podría escribir `C0 AF`) no la vería. Probémoslo con nuestro decodificador y con el de Python.

In [ ]:
for b in (b"\xc0\x80", b"\xc0\xaf", b"\xed\xa0\x80", b"\xf4\x90\x80\x80", b"\xe2\x82"):
    try: r = repr(utf8_decode(b))
    except ValueError as e: r = f"✗ {e}"
    try: p = repr(b.decode("utf-8"))
    except UnicodeDecodeError as e: p = f"✗ {e.reason}"
    print(b.hex(" ").upper().ljust(12), "| mío:", r.ljust(55)[:55], "| Python:", p)

## 4. UTF-16, UTF-32, *surrogates* y cómo guarda Python un `str`

* **UTF-32**: 4 bytes por code point, simple y desperdiciado.
* **UTF-16**: 2 bytes por code point del plano básico (BMP, `U+0000…U+FFFF`) y **4** (un *par sustituto*) para el resto. Fórmula: si $v = cp - 10000_{16}$, entonces $\text{alto}=D800_{16}+(v \gg 10)$ y $\text{bajo}=DC00_{16}+(v \,\&\, 3FF_{16})$. Por eso los *surrogates* no son caracteres válidos: son la «codificación» de UTF-16.
* **BOM** (`U+FEFF`): marca de orden de bytes al inicio de un archivo UTF-16; en UTF-8 es innecesario y suele ser un *bug* (aparece como un carácter invisible pegado a la primera palabra).
* **JavaScript** y Java miden `length` en **unidades UTF-16**: `"😀".length === 2`. Go y Rust miden `len` en **bytes**. Python mide **code points**. Truncar «a 100 caracteres» en cada lenguaje corta en sitios distintos.
* **CPython** (PEP 393) guarda cada `str` con **1, 2 o 4 bytes por code point** según el code point más grande que contenga.

In [ ]:
def surrogados(cp):
    v = cp - 0x10000
    return 0xD800 + (v >> 10), 0xDC00 + (v & 0x3FF)
alto, bajo = surrogados(ord("😀"))
print(f"😀 U+1F600 → UTF-16: {alto:04X} {bajo:04X}  | Python: {'😀'.encode('utf-16-be').hex(' ').upper()}")
assert f"{alto:04X}{bajo:04X}" == "😀".encode("utf-16-be").hex().upper()

filas = []
for nombre, ch in [("ASCII 'a'", "a"), ("Latin-1 'é'", "é"), ("BMP 'ā'", "ā"), ("BMP '€'", "€"), ("Plano 1 '😀'", "😀")]:
    s = ch * 1000
    filas.append({"carácter": nombre, "UTF-8 (bytes)": len(s.encode("utf-8")), "UTF-16 (bytes)": len(s.encode("utf-16-le")),
                  "UTF-32 (bytes)": len(s.encode("utf-32-le")), "RAM de str en CPython": sys.getsizeof(s)})
display(pd.DataFrame(filas).set_index("carácter"))

> 👀 **Qué debes observar.** La RAM de `str` en CPython salta a ≈ 1 B/carácter para ASCII y Latin-1, ≈ 2 B para BMP y ≈ 4 B en cuanto aparece **un** emoji: un solo carácter «grande» en un texto largo multiplica por 4 la memoria de **todo** el `str`. Y fíjate en que UTF-8 es la más compacta para ASCII pero la más cara que UTF-16 para `€` (3 vs 2 bytes): ninguna codificación es óptima para todos los idiomas.

## 5. Grafemas: lo que el lector percibe como «un carácter»

Un **grafema** (*extended grapheme cluster*, UAX #29) agrupa code points que se muestran juntos: una letra base + marcas combinantes, banderas (dos *regional indicators*), emojis con modificadores de tono de piel, secuencias unidas con **ZWJ** (U+200D), teclas (`1️⃣`), sílabas devanagari con conjuntos consonánticos… La librería `regex` implementa `\X`.

> ⚠️ La segmentación de grafemas **depende de la versión de Unicode** del runtime: un emoji nuevo puede partirse en dos grafemas si tu librería es antigua. Anota la versión (`regex` y `unicodedata.unidata_version`) cuando midas.

In [ ]:
casos = {"a": "a", "é (NFC)": "é", "é (NFD)": "é", "bandera ES": "🇪🇸", "pulgar tono medio": "👍🏽", "familia": "👨‍👩‍👧‍👦",
         "corazón (VS16)": "❤️", "tecla 1": "1️⃣", "bandera arcoíris": "🏳️‍🌈", "hangul (sílaba)": "한", "devanagari (conjunto)": "क्षि"}
filas = []
for nombre, s in casos.items():
    filas.append({"cadena": nombre, "grafemas": len(regex.findall(r"\X", s)), "code points (len)": len(s),
                  "unidades UTF-16": len(s.encode("utf-16-le")) // 2, "bytes UTF-8": len(s.encode("utf-8"))})
df_g = pd.DataFrame(filas).set_index("cadena")
display(df_g)
ax = df_g.plot.bar(figsize=(10, 4), color=[VERDE, AZUL, NARANJA, GRIS], width=0.8, rot=30)
ax.set_ylabel("unidades"); ax.set_title("Una misma cadena, cuatro formas de contarla"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La columna de **grafemas** es 1 en casi todo (y 1 también para `é` en las dos formas); las demás divergen hasta **25 bytes** en la familia. Consecuencias prácticas: contar «caracteres» para un límite de UI debe hacerse en grafemas; contar *costo* de un LLM se hace en tokens, que dependen de los **bytes** (módulo 02). Y `é (NFD)` demuestra que *el mismo grafema* puede tener 2 representaciones distintas: es el problema que resuelve la normalización.

## 6. Normalización Unicode: NFC, NFD, NFKC, NFKD

### El problema
Unicode permite **varias secuencias de code points para el mismo texto visible**: `é` puede ser `U+00E9` (1 code point, **precompuesto**) o `e` + `U+0301` (2 code points, **descompuesto**). Visualmente idénticos; para un programa, cadenas distintas: `"é" == "é"` es `False`, tienen distinto número de bytes y **producen distintos tokens**.

### Dos tipos de equivalencia y cuatro formas
* **Equivalencia canónica**: misma apariencia y significado (`é` ≡ `e`+◌́; también el signo `Å` ANGSTROM U+212B ≡ `Å` U+00C5).
* **Equivalencia de compatibilidad**: parecidas pero con distinción de formato/semántica (`ﬁ`≈`fi`, `²`≈`2`, `①`≈`1`, `Ｈ`≈`H`, `㎏`≈`kg`); aplanarlas **pierde información**.

| Forma | Qué hace | Cuándo |
|---|---|---|
| **NFD** | descompone canónicamente (y ordena las marcas combinantes) | análisis; quitar acentos (descartar marcas `Mn`) |
| **NFC** | NFD + recomposición canónica | **la forma «de intercambio»**: web, bases de datos; es la que recomienda W3C para texto |
| **NFKD** | descompone también por compatibilidad | búsqueda *agresiva* |
| **NFKC** | NFKD + recomposición | normalización de identificadores y de *matching*; SentencePiece usa una variante por defecto |

Algoritmo (UAX #15): **(1)** descomponer cada carácter con su tabla (recursivamente; las sílabas Hangul se descomponen *algorítmicamente* en *jamo*); **(2)** reordenar las marcas combinantes por su *combining class* (de modo estable); **(3)** (NFC/NFKC) recomponer pares evitando las *exclusiones de composición*. Implementamos (1)+(2) —**NFD**— a mano usando las tablas de `unicodedata` y lo verificamos contra la implementación de Python en **todo** el espacio Unicode.

> ### 🎨 GEMINI-IMAGEN · ID: `M01-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Las cuatro formas de normalización**
>
> **Qué muestra:** Un diagrama de las cuatro formas (NFC, NFD, NFKC, NFKD) con las dos operaciones (descomponer, recomponer) y ejemplos reales: é, Å (signo ångström) y la ligadura ﬁ.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuadrícula 2x2 con NFC, NFD, NFKC y NFKD y los ejemplos é, Å y la ligadura fi
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible, sin ilustraciones decorativas.
> Composición: una cuadrícula de 2×2 cajas grandes con flechas entre ellas.
> Fila superior (borde azul, etiqueta lateral «Equivalencia canónica»): a la izquierda la caja «NFC (compuesta)», a la derecha la caja «NFD (descompuesta)». Entre ambas, una flecha hacia la derecha etiquetada «descomponer» y una flecha hacia la izquierda etiquetada «recomponer».
> Fila inferior (borde naranja, etiqueta lateral «Equivalencia de compatibilidad»): a la izquierda la caja «NFKC (compuesta)», a la derecha la caja «NFKD (descompuesta)», con las mismas dos flechas entre ellas.
> Flechas verticales grises de cada caja superior a la inferior de su columna, etiquetadas «+ compatibilidad (pierde formato)».
> Dentro de cada caja, tres ejemplos con estos textos EXACTOS:
> NFC: «é = U+00E9», «Å (ångström U+212B) → U+00C5», «ﬁ = U+FB01 (se conserva)».
> NFD: «é = U+0065 U+0301», «Å → U+0041 U+030A», «ﬁ = U+FB01 (se conserva)».
> NFKC: «é = U+00E9», «Å → U+00C5», «ﬁ → fi (U+0066 U+0069)».
> NFKD: «é = U+0065 U+0301», «Å → U+0041 U+030A», «ﬁ → fi (U+0066 U+0069)».
> Título superior: «Formas de normalización Unicode». Nota inferior en gris: «NFC es la forma recomendada para intercambio de texto; NFKC/NFKD pierden información».
> Evita: cambiar los code points indicados, texto ilegible, flechas ambiguas, iconos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M01-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuadrícula 2x2 con NFC, NFD, NFKC y NFKD y los ejemplos é, Å y la ligadura fi](assets/M01-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
>                    descomponer →
> canónica       [ NFC ]  ──────────►  [ NFD ]
>                é = U+00E9            é = U+0065 U+0301
>                Å(U+212B)→U+00C5      Å → U+0041 U+030A
>                ﬁ = U+FB01            ﬁ = U+FB01
>                    ◄── recomponer
>        + compatibilidad ↓                 ↓
> compat.        [ NFKC ] ─────────►  [ NFKD ]
>                ﬁ → fi (U+0066 U+0069)  ﬁ → fi
> ```

> 👀 **Qué debes observar.** Mira la **fila canónica**: NFC y NFD son intercambiables *sin pérdida*. En la **fila de compatibilidad**, `ﬁ` se convierte en `fi` y ya no hay vuelta atrás (por eso NFKC es útil para *buscar* pero peligroso para *almacenar* o para código).

In [ ]:
ejemplos = ["é", "é", "Å", "ﬁ", "²", "①", "Ｈｅｌｌｏ", "㎏", "½", "한"]
filas = []
for s in ejemplos:
    fila = {"original": s, "code points": " ".join(f"{ord(c):04X}" for c in s)}
    for forma in ("NFC", "NFD", "NFKC", "NFKD"):
        n = unicodedata.normalize(forma, s)
        fila[forma] = f"{n} ({' '.join(f'{ord(c):04X}' for c in n)})"
    filas.append(fila)
display(pd.DataFrame(filas).set_index("original"))
print("é == e+◌́ ?", "é" == "é", "| tras NFC:", unicodedata.normalize("NFC", "é") == unicodedata.normalize("NFC", "é"))

In [ ]:
# NFD «a mano»: descomposición canónica recursiva + Hangul algorítmico + orden canónico de marcas combinantes
SBASE, LBASE, VBASE, TBASE, LCOUNT, VCOUNT, TCOUNT = 0xAC00, 0x1100, 0x1161, 0x11A7, 19, 21, 28
NCOUNT = VCOUNT * TCOUNT; SCOUNT = LCOUNT * NCOUNT

def _decomp_canonica(ch: str) -> str:
    cp = ord(ch)
    if SBASE <= cp < SBASE + SCOUNT:                              # sílaba Hangul → jamo (algoritmo, sin tabla)
        s = cp - SBASE
        partes = [LBASE + s // NCOUNT, VBASE + (s % NCOUNT) // TCOUNT] + ([TBASE + s % TCOUNT] if s % TCOUNT else [])
        return "".join(map(chr, partes))
    d = unicodedata.decomposition(ch)
    if not d or d.startswith("<"):                                # sin descomposición, o de compatibilidad (<font>, <compat>…)
        return ch
    return "".join(_decomp_canonica(chr(int(h, 16))) for h in d.split())

def mi_nfd(s: str) -> str:
    out = [c for ch in s for c in _decomp_canonica(ch)]
    i = 0
    while i < len(out):                                           # ordena cada racha de marcas por su combining class (estable)
        if unicodedata.combining(out[i]) == 0:
            i += 1; continue
        j = i
        while j < len(out) and unicodedata.combining(out[j]) != 0:
            j += 1
        out[i:j] = sorted(out[i:j], key=unicodedata.combining)
        i = j
    return "".join(out)

t0 = time.time()
malos = [cp for cp in range(0x110000) if not 0xD800 <= cp <= 0xDFFF and mi_nfd(chr(cp)) != unicodedata.normalize("NFD", chr(cp))]
rng = random.Random(SEED); marcas = [chr(c) for c in range(0x300, 0x370)] + ["̣", "̛"]; base = list("aeouñ") + ["ệ", "ǖ", "ṩ", "한", "ǆ"]
fallos = sum(mi_nfd(s) != unicodedata.normalize("NFD", s) for s in
             ("".join(rng.choice(base) + "".join(rng.choice(marcas) for _ in range(rng.randint(0, 3))) for _ in range(3)) for _ in range(20_000)))
print(f"mi_nfd vs unicodedata: {0x110000 - 2048:,} caracteres sueltos → {len(malos)} discrepancias; 20 000 cadenas con marcas desordenadas → {fallos} discrepancias ({time.time() - t0:.1f} s)")
print("ṩ  (s + punto debajo + punto encima) en orden inverso →", [f"{ord(c):04X}" for c in mi_nfd("ṩ")], "(el punto de abajo, ccc=220, pasa antes que el de arriba, ccc=230)")
assert not malos and fallos == 0

### ¿Qué normalizan los tokenizers de verdad?
La normalización es **la primera etapa** de un pipeline de tokenización (módulo 03 lo desarrolla con `tokenizers`):

* **`tiktoken`** (GPT-2, cl100k, o200k) **no normaliza**: toma los bytes tal cual (lo verificamos abajo con la librería).
* **SentencePiece** aplica por defecto la regla **`nmt_nfkc`** (NFKC + normalización de espacios); la regla `identity` la desactiva (documentación oficial de SentencePiece).
* **BERT *uncased*** (en Hugging Face `BertTokenizer`) pasa a minúsculas y **elimina acentos** vía NFD + descartar marcas `Mn` (parámetro `strip_accents`).
* **Hugging Face `tokenizers`** ofrece normalizadores componibles (`NFC`, `NFD`, `NFKC`, `NFKD`, `Lowercase`, `StripAccents`, `BertNormalizer`…), con el mismo resultado que `unicodedata`.

In [ ]:
from tokenizers import normalizers
muestra = "Canción ﬁnal ① Ｈｏｌａ"
for nombre, norm in [("NFC", normalizers.NFC()), ("NFD", normalizers.NFD()), ("NFKC", normalizers.NFKC()),
                     ("NFD+StripAccents", normalizers.Sequence([normalizers.NFD(), normalizers.StripAccents()]))]:
    r = norm.normalize_str(muestra)
    print(f"{nombre:18}→ {r!r}  (len = {len(r)} code points)")
assert normalizers.NFD().normalize_str("canción") == mi_nfd("canción") == unicodedata.normalize("NFD", "canción")
print("✔ HF tokenizers NFD == mi_nfd == unicodedata NFD")

### Impacto 1: bytes y tokens
Un tokenizer **byte-level** (el de GPT-2) ve los bytes. Misma palabra, dos formas: ¿cuántos bytes y cuántos tokens cuesta cada una? Usamos el vocabulario real de GPT-2 (descargado de `tiktoken` o de un espejo de GitHub verificado con SHA-256).

In [ ]:
import hashlib
import os
import tempfile
import urllib.request


def get_gpt2_encoding():
    """tiktoken 'gpt2'. 1) oficial (openaipublic); 2) espejo en GitHub (graykode/gpt-2-Pytorch) verificado con los
    mismos SHA-256 que exige tiktoken; 3) None (las celdas que lo usan se omiten con un aviso)."""
    import tiktoken
    try:
        return tiktoken.get_encoding("gpt2")
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] descarga oficial no disponible ({type(e).__name__}); pruebo el espejo de GitHub...")
    try:
        import tiktoken_ext.openai_public as op
        from tiktoken.load import data_gym_to_mergeable_bpe_ranks
        base = "https://raw.githubusercontent.com/graykode/gpt-2-Pytorch/master/GPT2/"
        sha = {"vocab.bpe": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
               "encoder.json": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"}
        paths = {}
        for name, h in sha.items():
            data = urllib.request.urlopen(base + name, timeout=60).read()
            assert hashlib.sha256(data).hexdigest() == h, f"hash de {name} no coincide"
            paths[name] = os.path.join(tempfile.gettempdir(), "gpt2_" + name)
            with open(paths[name], "wb") as f:
                f.write(data)
        ranks = data_gym_to_mergeable_bpe_ranks(vocab_bpe_file=paths["vocab.bpe"], encoder_json_file=paths["encoder.json"])
        print("[tiktoken] GPT-2 cargado desde el espejo de GitHub (hashes verificados)")
        return tiktoken.Encoding("gpt2", pat_str=op.r50k_pat_str, mergeable_ranks=ranks,
                                 special_tokens={"<|endoftext|>": 50256})
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] sin GPT-2 ({type(e).__name__}: {str(e)[:80]}). Las comparaciones con tiktoken se omiten.")
        return None

In [ ]:
enc = get_gpt2_encoding()
palabras = ["canción", "número", "información", "más", "año", "niño", "pingüino", "también", "país", "señal"]
filas = []
for w in palabras:
    nfc, nfd = unicodedata.normalize("NFC", w), unicodedata.normalize("NFD", w)
    filas.append({"palabra": w, "bytes NFC": len(nfc.encode()), "bytes NFD": len(nfd.encode()),
                  "tokens NFC": len(enc.encode(nfc)) if enc else np.nan, "tokens NFD": len(enc.encode(nfd)) if enc else np.nan})
df_n = pd.DataFrame(filas).set_index("palabra")
display(df_n)
if enc:
    ax = df_n[["tokens NFC", "tokens NFD"]].plot.bar(figsize=(9, 3.8), color=[AZUL, NARANJA], rot=30, width=0.8)
    ax.set_ylabel("tokens GPT-2"); ax.set_title("La misma palabra visible, dos normalizaciones, dos costos")
    plt.tight_layout(); plt.show()
    print("media de tokens NFC → NFD:", df_n["tokens NFC"].mean(), "→", df_n["tokens NFD"].mean())
    print("tiktoken NO normaliza: encode('é') =", enc.encode("é"), "| encode('e' + U+0301) =", enc.encode("e\u0301"))
    assert enc.encode("é") != enc.encode("e\u0301")

> 👀 **Qué debes observar.** Para **cada** palabra, la forma NFD cuesta más bytes (+1 por marca combinante) y **más tokens** (el vocabulario de GPT-2 contiene tokens precompuestos como `ón`, `ás` o `ño` (mira `canción` → `can`·`ci`·`ón`) pero casi ninguno con la marca `U+0301` suelta, así que `o`+◌́ cae a bytes). Si tu corpus mezcla NFC y NFD, el mismo concepto genera **secuencias de tokens distintas**: peor compresión, vocabulario fragmentado y *recall* perdido.

### Impacto 2: búsqueda
Volvamos a Políglota. Indexamos los documentos en español con un tokenizador léxico *ingenuo* (`\w+` en minúsculas, **sin** normalizar) y lanzamos las consultas en NFC y en NFD. Después probamos con la normalización NFC dentro del tokenizador. Reutilizamos `carga_corpus`, las métricas y BM25 del módulo 00.

In [ ]:
# ── Utilidades canónicas del módulo 00 (1/3): datos del corpus sintético ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Utilidades canónicas del módulo 00 (2/3): generador sintético ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── Utilidades canónicas del módulo 00 (3/3): carga_corpus ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas canónicas del módulo 00 ──
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── BM25 del módulo 00 ──
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels, source = corpus["docs"], corpus["queries"], corpus["qrels"], corpus["source"]
print(f"source = {source!r} | {len(docs)} docs | {len(queries)} queries")

es_docs = [d for d in docs if d["lang"] == "es"]; es_q = [q for q in queries if q["lang"] == "es"]
qr_es = {q["qid"]: qrels[q["qid"]] for q in es_q}
ingenuo = lambda t: re.findall(r"\w+", t.lower())
nfc = lambda t: unicodedata.normalize("NFC", t); nfd = lambda t: unicodedata.normalize("NFD", t)

def ndcg_es(tok, f_doc=lambda t: t, f_q=lambda t: t):
    bm = BM25().fit([tok(f_doc(d["text"])) for d in es_docs], [d["id"] for d in es_docs])
    run = {q["qid"]: bm.search(tok(f_q(q["text"])), 10) for q in es_q}
    return evaluate(run, qr_es)["ndcg"]

res = {"docs NFC · consulta NFC": ndcg_es(ingenuo), "docs NFC · consulta NFD": ndcg_es(ingenuo, f_q=nfd),
       "docs NFD · consulta NFC": ndcg_es(ingenuo, f_doc=nfd), "docs NFD · consulta NFD": ndcg_es(ingenuo, nfd, nfd),
       "docs NFC · consulta NFD\n(+ NFC en el tokenizador)": ndcg_es(lambda t: ingenuo(nfc(t)), f_q=nfd)}
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(list(res), list(res.values()), color=[VERDE, ROJO, ROJO, VERDE, AZUL]); ax.set_ylim(0, 1.05)
for i, v in enumerate(res.values()): ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
ax.set_ylabel("nDCG@10 (consultas ES)"); ax.set_title(f"Efecto de la forma de normalización en la búsqueda (corpus {source})")
plt.xticks(fontsize=8); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Cuando la forma de la consulta **no coincide** con la del índice (barras rojas) el nDCG cae, porque las palabras acentuadas dejan de coincidir como cadenas. Cuando ambas coinciden (verdes) o el tokenizador normaliza (azul), se recupera. La caída depende de la fracción de consultas con tildes en tu corpus (sintético: pequeña; Wikipedia real: mayor). **Regla:** normaliza *al ingerir* y *al consultar* con la misma función.

> **🧪 Experimento.** **Python normaliza tus identificadores con NFKC** (PEP 3131). Puedes verificarlo: `ａ = 1` (con «ａ» de ancho completo) define la variable `a`.

In [ ]:
ns = {}
exec("ａ = 1\n", ns)
print("variables definidas:", [k for k in ns if not k.startswith("__")], "| 'a' in ns:", "a" in ns)

## 7. Homoglifos, caracteres invisibles y seguridad

Dos cadenas pueden **verse idénticas** y ser distintas porque Unicode contiene caracteres de aspecto igual en escrituras diferentes (**homoglifos**: la «а» cirílica U+0430 y la «a» latina U+0061) y caracteres **sin apariencia** (espacio de ancho cero U+200B, unión de ancho cero U+200D, BOM U+FEFF, **controles bidireccionales** U+202A–202E y U+2066–2069, y los *tag characters* U+E0000–E007F).

* **Homoglifos** → ataques de *phishing* (IDN homograph), cuentas suplantadas, evasión de filtros.
* **Controles bidireccionales** → *Trojan Source* (Boucher & Anderson, 2021): el código se **ve** de una manera y se **ejecuta** de otra.
* **Tag characters** → «ASCII smuggling»: texto invisible que los modelos pueden leer. Documentado en 2024 por Johann Rehberger contra asistentes con LLM (ver referencias).

> ### 🎨 GEMINI-IMAGEN · ID: `M01-IMG-04` · ⚠️ REEMPLAZAR ESTA CELDA
> **Homoglifos: mismas letras para el ojo, distintos bytes**
>
> **Qué muestra:** La palabra «paypal» auténtica frente a «pаypal» con una «а» cirílica: idéntica a la vista, distinta en code points y bytes. Muestra por qué un tokenizer y un filtro ven cosas distintas.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Dos filas comparando paypal en latino y con una a cirílica: code points y bytes distintos
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB para la versión latina, naranja #F28E2B para la versión con el carácter cirílico, rojo sobrio #D62728 solo para resaltar la diferencia, gris #6B7280 para anotaciones. Tipografía sans-serif legible y monoespaciada para los códigos, sin ilustraciones decorativas.
> Composición: dos filas horizontales con la misma estructura, una sobre otra, con una línea de comparación entre ellas.
> Fila superior (borde azul), título «Todo latino». Muestra la palabra «paypal» en grande; debajo los code points EXACTOS: «U+0070 U+0061 U+0079 U+0070 U+0061 U+006C»; debajo los bytes EXACTOS: «70 61 79 70 61 6C» (6 bytes).
> Fila inferior (borde naranja), título «Con una «а» cirílica en la 2ª posición». Muestra «pаypal» en grande, con la segunda letra marcada con un círculo rojo; debajo los code points EXACTOS: «U+0070 U+0430 U+0079 U+0070 U+0061 U+006C»; debajo los bytes EXACTOS: «70 D0 B0 79 70 61 6C» (7 bytes), con «D0 B0» resaltado en rojo.
> Entre las filas, un signo «≠» grande con la etiqueta «Se ven igual, son cadenas distintas».
> A la derecha, un cuadro gris con el texto «Defensa: detectar palabras con escrituras mezcladas (LATIN + CYRILLIC) y normalizar a un esqueleto».
> Título superior: «Homoglifos».
> Evita: cambiar los códigos o bytes, texto ilegible, elementos decorativos o alarmistas, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M01-IMG-04.png` en la carpeta del módulo y reemplaza esta celda por: `![Dos filas comparando paypal en latino y con una a cirílica: code points y bytes distintos](assets/M01-IMG-04.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Todo latino          paypal     U+0070 U+0061 U+0079 U+0070 U+0061 U+006C    70 61 79 70 61 6C        (6 bytes)
>                        ≠   se ven igual, son cadenas distintas
> Con 'а' cirílica     pаypal     U+0070 U+0430 U+0079 U+0070 U+0061 U+006C    70 D0 B0 79 70 61 6C     (7 bytes)
> ```

> 👀 **Qué debes observar.** Compara las dos filas: la diferencia está en **una** posición (`61` → `D0 B0`), pero esa diferencia cambia la longitud en bytes (6 → 7), los tokens de un tokenizer byte-level y el resultado de cualquier comparación exacta.

In [ ]:
def escritura(ch: str) -> str:
    """Escritura aproximada de un carácter por la primera palabra de su nombre Unicode (LATIN, CYRILLIC, GREEK…)."""
    n = unicodedata.name(ch, "")
    return n.split()[0] if n else "?"

def palabra_mixta(w: str) -> bool:
    return len({escritura(c) for c in w if c.isalpha()} & {"LATIN", "CYRILLIC", "GREEK"}) >= 2

limpio, sucio = "paypal", "pаypal"
print(limpio == sucio, "| mixta:", palabra_mixta(limpio), palabra_mixta(sucio), "| escrituras:", {escritura(c) for c in sucio})
for s in ("a​b", "ho​la", "‮odnum", "A\U000e0042"):
    print(repr(s), "→ tokens GPT-2:", len(enc.encode(s)) if enc else "n/d", "| visibles:", len(s.encode("ascii", "ignore")), "ASCII")
ocultos = "".join(chr(0xE0000 + ord(c)) for c in "hi")
print("texto oculto en tag characters:", repr(ocultos), "| decodificado:", "".join(chr(ord(c) - 0xE0000) for c in ocultos))

> 👀 **Qué debes observar.** Un único `U+200B` entre dos letras **cambia el número de tokens**: el tokenizer lo ve (no es «nada»). Por eso la normalización de Políglota (proyecto) elimina los invisibles de ancho cero y los controles bidireccionales, y marca las palabras con escrituras mezcladas. Los *tag characters* ocultan texto que no se ve pero que sí está en la secuencia.

## 8. Bytes por carácter según el idioma (y su costo posterior)

La razón por la que el español pagó más tokens que el inglés en el módulo 00 empieza aquí. Medimos la **misma frase** («Hola, ¿cómo estás hoy?») en 12 idiomas: code points, grafemas, bytes UTF-8 y tokens GPT-2. *(Son frases cortas escritas a mano, para ilustrar, no para medir: la comparación rigurosa con Flores-200 es el módulo 04.)*

In [ ]:
frases = {
    "inglés": "Hello, how are you today?", "español": "Hola, ¿cómo estás hoy?", "francés": "Bonjour, comment allez-vous aujourd'hui ?",
    "alemán": "Hallo, wie geht es dir heute?", "ruso": "Привет, как дела сегодня?", "griego": "Γεια σου, πώς είσαι σήμερα;",
    "árabe": "مرحبًا، كيف حالك اليوم؟", "hindi": "नमस्ते, आज आप कैसे हैं?", "tailandés": "สวัสดี วันนี้คุณเป็นอย่างไรบ้าง",
    "chino": "你好，你今天好吗？", "japonés": "こんにちは、今日はお元気ですか？", "coreano": "안녕하세요, 오늘 어떻게 지내세요?",
}
filas = []
for idioma, s in frases.items():
    filas.append({"idioma": idioma, "code points": len(s), "grafemas": len(regex.findall(r"\X", s)), "bytes UTF-8": len(s.encode("utf-8")),
                  "bytes/code point": len(s.encode("utf-8")) / len(s), "tokens GPT-2": len(enc.encode(s)) if enc else np.nan})
df_i = pd.DataFrame(filas).set_index("idioma")
df_i["tokens/code point"] = df_i["tokens GPT-2"] / df_i["code points"]
display(df_i.round(2))
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
df_i["bytes/code point"].sort_values().plot.barh(ax=axs[0], color=AZUL); axs[0].set_xlabel("bytes por code point"); axs[0].set_title("Costo en bytes")
if enc:
    df_i["tokens GPT-2"].sort_values().plot.barh(ax=axs[1], color=NARANJA); axs[1].set_xlabel("tokens GPT-2 por frase"); axs[1].set_title("Costo en tokens (vocabulario GPT-2)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. **Bytes por code point** (columna izquierda): inglés, francés y alemán ≈ 1.00 (ASCII); español 1.14 por «¿», «ó», «á»; ruso, griego y árabe ≈ 1.8; hindi y coreano ≈ 2.5; tailandés ≈ 2.9; chino y japonés 3.0. Es el **piso** de lo que paga un tokenizer byte-level.
> 2. **Tokens GPT-2** (columna derecha, vocabulario fijo de GPT-2 verificado con `tiktoken`): el patrón se **amplifica**. La frase en inglés cuesta 7 tokens y la española 13; la del tailandés, **61**; la del coreano, 43. Con un vocabulario aprendido sobre todo en inglés, las escrituras poco vistas caen a bytes sueltos (casi un token por byte). Esta desigualdad la documentan Petrov et al. (2023) y Ahia et al. (2023).
> 3. Los *grafemas* difieren de los code points en árabe, hindi y tailandés (marcas combinantes): contar «caracteres» en esas escrituras es ambiguo.
> 4. Son frases cortas escritas a mano: busca la **tendencia**, no la cifra exacta (la comparación rigurosa con Flores es el módulo 04).

### Analogía lingüística: ¿qué unidad representa un carácter?
El costo en bytes no es arbitrario: depende de **qué cubre un code point** en cada sistema de escritura.

| Tipo de escritura | Ejemplo | Un code point ≈ | Bytes UTF-8 típicos |
|---|---|---|---|
| Alfabeto (latino, cirílico, griego) | español, ruso | un fonema | 1–2 |
| *Abjad* (consonantes) | árabe | una consonante (vocales como marcas) | 2 |
| *Abugida* (consonante+vocal) | hindi, tailandés | consonante o signo vocálico (conjuntos con marcas) | 3 |
| Silabario / sílabas compuestas | kana japonés, hangul coreano | una sílaba (hangul: bloque precompuesto) | 3 |
| Logografía | chino (hanzi) | una palabra o morfema | 3 |

Un carácter chino «vale» mucho más información que una `e`, pero cuesta 3 bytes. **Los bytes no miden información**; para eso hace falta *comprimir* (próxima sección).

## 9. Analogía con compresión de datos: bits por byte

Un byte tiene 8 bits pero **no aporta 8 bits de información**. La entropía de orden 0 de la distribución de bytes de un texto, $H_0=-\sum_{b}p(b)\log_2 p(b)$ (bits por byte), es un **límite inferior** para cualquier código que codifique cada byte por separado (Shannon, 1948); un compresor que explota el contexto (`zlib`/DEFLATE = LZ77 + Huffman, `bz2`, `lzma`) baja mucho más. En el módulo 02 verás que **BPE nació como compresor** (Gage, 1994) y que un tokenizer es, en el fondo, un compresor con un vocabulario fijo; la métrica natural es *bits por byte* de texto original.

In [ ]:
def bits_por_byte(texto: str) -> dict:
    import bz2, lzma, zlib
    b = texto.encode("utf-8"); n = len(b); cnt = collections.Counter(b)
    H = -sum(v / n * math.log2(v / n) for v in cnt.values())
    return {"bytes": n, "H0 (orden 0)": H, "zlib": len(zlib.compress(b, 9)) * 8 / n, "bz2": len(bz2.compress(b)) * 8 / n, "lzma": len(lzma.compress(b)) * 8 / n}

filas = {lang: bits_por_byte("\n".join(d["text"] for d in docs if d["lang"] == lang)) for lang in ("es", "en", "code")}
df_c = pd.DataFrame(filas).T
display(df_c.round(3))
ax = df_c[["H0 (orden 0)", "zlib", "bz2", "lzma"]].plot.bar(figsize=(8, 3.8), color=[GRIS, AZUL, NARANJA, VERDE], rot=0, width=0.8)
ax.axhline(8, color=ROJO, ls="--", lw=1); ax.text(2.55, 7.7, "8 bits (sin comprimir)", color=ROJO, ha="right", fontsize=8)
ax.set_ylabel("bits por byte de texto"); ax.set_title(f"Entropía y compresión del corpus Políglota ({source})")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. $H_0$ ≈ **4.5 bits/byte** en los tres corpus de esta ejecución: mirando solo la frecuencia de cada byte, un byte «vale» bastante menos de los 8 bits que ocupa.
> 2. Los compresores con contexto bajan **mucho** más. Con el corpus **sintético** el resultado es exagerado porque es muy repetitivo (plantillas): con Wikipedia real verás cifras más altas. Lo importante: *comprimir = capturar regularidades de más largo alcance que el byte*, exactamente lo que hará BPE al fusionar pares frecuentes.

## 10. Mojibake y manejadores de errores

**Mojibake** (文字化け) es texto cuyos bytes se **decodificaron con la codificación equivocada**. El caso típico en español: UTF-8 leído como Latin-1/cp1252, que convierte `é` (`C3 A9`) en `Ã©`.

In [ ]:
original = "Canción: «niño» – ¿qué señal?"
for enc_mal in ("latin-1", "cp1252"):
    roto = original.encode("utf-8").decode(enc_mal, errors="replace")
    print(f"{enc_mal:7}→ {roto}")
roto = original.encode("utf-8").decode("latin-1")
print("reparado :", roto.encode("latin-1").decode("utf-8"))
doble = roto.encode("utf-8").decode("latin-1")
print("doble mojibake:", doble[:40], "…")

import ftfy
print("ftfy.fix_text:", ftfy.fix_text(roto), "|", ftfy.fix_text(doble))

In [ ]:
# Manejadores de errores de codificación: qué pasa con bytes que no son UTF-8 válido
crudo = b"abc\xff\xfedef"
for manejador in ("strict", "replace", "ignore", "backslashreplace", "surrogateescape"):
    try:
        s = crudo.decode("utf-8", errors=manejador)
        ida_vuelta = s.encode("utf-8", errors=manejador) == crudo if manejador in ("surrogateescape", "backslashreplace") else "—"
        print(f"{manejador:17}→ {s!r:28} ¿recupera los bytes originales? {ida_vuelta}")
    except UnicodeDecodeError as e:
        print(f"{manejador:17}→ ✗ UnicodeDecodeError ({e.reason})")

# Decodificación incremental: un token byte-level puede terminar a mitad de un carácter UTF-8
import codecs
dec = codecs.getincrementaldecoder("utf-8")()
trozos = "añ😀".encode("utf-8")
print("\nflujo byte a byte:", [dec.decode(trozos[i:i + 1]) for i in range(len(trozos))])

> 👀 **Qué debes observar.** `replace` y `ignore` **destruyen información sin avisar** (el mojibake con `�` ya no es reparable); `surrogateescape` permite ida y vuelta sin pérdida de bytes desconocidos. Y el decodificador **incremental** muestra por qué, al hacer *streaming* de un LLM byte-level, a veces el siguiente token «no imprime nada»: sus bytes son solo parte de un carácter. Lo retomarás en el módulo 02.

## 11. Políglota: ¿qué escrituras contiene tu corpus?
Cerramos la lección mirando el corpus del módulo 00: ¿qué escrituras aparecen y cuántos bytes por carácter cuesta cada tipo de documento? (En el proyecto harás la auditoría completa y un normalizador.)

In [ ]:
def cuenta_escrituras(textos):
    cnt = collections.Counter()
    for t in textos:
        for ch, n in collections.Counter(t[:3000]).items():
            if ch.isalpha():
                cnt[escritura(ch)] += n
    return cnt

por_lang = {lang: cuenta_escrituras(d["text"] for d in docs if d["lang"] == lang) for lang in ("es", "en", "code")}
df_s = pd.DataFrame(por_lang).fillna(0)
df_s = df_s.loc[df_s.sum(axis=1).sort_values(ascending=False).index[:8]]
frac = (df_s / df_s.sum()).T
bpc = {l: sum(len(d["text"].encode()) for d in docs if d["lang"] == l) / sum(len(d["text"]) for d in docs if d["lang"] == l) for l in ("es", "en", "code")}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
frac.plot.bar(stacked=True, ax=axs[0], rot=0, colormap="tab20"); axs[0].set_ylabel("fracción de letras"); axs[0].set_title(f"Escrituras por tipo de documento ({source})"); axs[0].legend(fontsize=7)
axs[1].bar(list(bpc), [(v - 1) * 100 for v in bpc.values()], color=[AZUL, NARANJA, VERDE]); axs[1].set_ylabel("bytes extra por cada 100 caracteres"); axs[1].set_title("Sobrecoste de UTF-8 sobre ASCII")
plt.tight_layout(); plt.show()
print({k: round(v, 4) for k, v in bpc.items()})

> 👀 **Qué debes observar.** Con datos **reales** (Wikipedia) verás, además de LATIN, trazas de CYRILLIC, GREEK, CJK y ARABIC en nombres propios dentro de los artículos; con el corpus sintético casi todo es LATIN. El sobrecoste de UTF-8 en español es del orden de **un uno por ciento** de bytes extra: pequeño en bytes, pero (como viste con GPT-2) mucho mayor en **tokens**, porque cada vocal con tilde rompe el patrón de las palabras más frecuentes.

## 12. Mapa de madurez
Estado a **octubre de 2026**. 🟢 sólido · 🟡 emergente · 🔴 experimental.


| Técnica | Madurez (oct-2026) | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| UTF-8 como codificación universal de texto | 🟢 | Estándar de intercambio; los tokenizers byte-level (GPT-2, `tiktoken`) operan sobre sus bytes | RFC 3629; Radford et al. 2019; Wang et al. 2019 (arXiv:1909.03341) |
| Normalización NFC/NFKC antes de tokenizar | 🟢 | SentencePiece: `nmt_nfkc` por defecto; BERT *uncased*: NFD + quitar acentos | Documentación de SentencePiece (`doc/normalization.md`); código de `BertTokenizer` en `transformers` |
| Tokenizar bytes sin normalizar (`tiktoken`) | 🟢 | Encodings de OpenAI (`gpt2`, `cl100k_base`, `o200k_base`) | Repositorio `openai/tiktoken` (código fuente de `Encoding.encode`) |
| Detección de confusables / escrituras mezcladas | 🟢 | Mecanismos de detección de suplantación (*spoofing*) definidos por el Consorcio Unicode | Unicode Technical Standard #39 (Unicode Security Mechanisms) |
| Defensa contra Trojan Source y caracteres bidi | 🟢 | Mitigaciones en compiladores y editores descritas tras la divulgación coordinada (2021) | Boucher & Anderson 2021 (arXiv:2111.00169) |
| Filtrado de *tag characters* y texto invisible en prompts de LLM | 🟡 | Reportado contra asistentes con LLM en 2024; las defensas varían por proveedor | Cobertura de The Register (ago-2024) sobre el trabajo de J. Rehberger |
| Modelos que consumen bytes directamente (ByT5, BLT) | 🟡 / 🔴 | Investigación de Google (ByT5) y Meta FAIR (BLT); no son el estándar de producción | Xue et al. 2021 (arXiv:2105.13626); Pagnoni et al. 2024 (arXiv:2412.09871) |

## 13. 🏭 En producción
* **Normaliza en la frontera.** En el pipeline de ingesta de un buscador, normaliza a **NFC** (y aplica la misma función a las consultas), conserva el original para mostrarlo y **guarda la versión de la normalización** junto al índice: cambiarla invalida los índices.
* **Búsqueda léxica con plegado.** Elasticsearch/OpenSearch ofrecen filtros `asciifolding` y, con el plugin ICU, `icu_normalizer`/`icu_folding` para hacer la normalización y el plegado de acentos en el analizador (es el mismo problema que acabas de resolver a mano).
* **Costo de tokens por idioma.** Los proveedores cobran por token; el sobrecoste de bytes de una escritura se traduce en dinero y en menos contexto útil (Petrov et al. 2023; Ahia et al. 2023). Mide *tokens por carácter* de **tu** tráfico real antes de elegir proveedor.
* **Seguridad de entrada.** Antes de pasar texto de usuarios a un LLM: normaliza, elimina controles bidi y tag characters, y registra escrituras mezcladas. Es barato y bloquea una familia entera de ataques.
* **Truncar por grafemas.** Para mostrar «los primeros 100 caracteres» en UI, corta en grafemas (`\X`), no en code points ni en bytes: evitas emojis partidos y marcas huérfanas.

## 14. 🧠 Secretos de la élite
1. **`len()` no significa lo mismo en todos los lenguajes**: Python cuenta code points, JavaScript/Java unidades UTF-16, Go/Rust bytes. Un límite de «N caracteres» validado en el *frontend* (JS) y en el *backend* (Python) puede discrepar en cualquier texto con emojis; compruébalo con la tabla de la sección 5.
2. **NFC y NFD producen secuencias de tokens distintas** en un tokenizer byte-level: mide la tabla de la sección 6 con tu propio tokenizer antes de entrenarlo. Un corpus de preentrenamiento sin normalizar fragmenta el vocabulario.
3. **NFKC es un martillo**: pliega `²`→`2`, `ﬁ`→`fi`, `㎏`→`kg` y rompe código/fórmulas. Úsalo para *matching* y para identificadores (Python lo aplica a los nombres de variables, PEP 3131; lo verificaste arriba) y no para almacenar texto.
4. **La versión de Unicode importa**: `unicodedata.unidata_version` y la de `regex` determinan categorías, normalización y segmentación de grafemas; un emoji nuevo puede partirse en dos en un runtime antiguo. Registra la versión en tus experimentos.
5. **Un solo carácter invisible cambia el número de tokens** (sección 7). Los ataques de *smuggling* explotan que el tokenizer ve lo que el humano no: sanea antes del tokenizer, no después.
6. **`errors="replace"` es destructivo y silencioso**: el `�` (U+FFFD) no es reversible. Para datos de origen desconocido usa `surrogateescape` y arregla la codificación en origen (`ftfy` solo es una red de seguridad).
7. **Un solo emoji multiplica por 4 la RAM de un `str` de CPython** (PEP 393, sección 4): relevante al procesar corpus enormes carácter a carácter en Python.
8. **Los token a mitad de carácter existen**: un token de un BPE byte-level puede ser medio carácter UTF-8; para *streaming* hace falta un decodificador incremental (sección 10) y es la causa de los `�` que a veces ves en salidas parciales.

## 15. ⚠️ Errores comunes
* **Usar `encode("ascii", "ignore")` para «quitar acentos»**: borra *todo* carácter no ASCII (ñ, 中, emojis). Lo correcto: NFD + descartar marcas `Mn` (y solo en escrituras latinas).
* **Comparar cadenas sin normalizar** (`"é" == "é"` es falso) en búsquedas, claves de diccionario o deduplicación.
* **Aplicar NFKC a todo** y romper código, fórmulas y nombres.
* **Abrir archivos sin `encoding="utf-8"`** (el valor por defecto depende de la plataforma) y heredar mojibake.
* **Cortar bytes a ciegas** (`s.encode()[:100].decode()`) y partir caracteres: usa `errors="ignore"` solo conscientemente o corta en límites de code point/grafema.
* **Ignorar caracteres de formato** (BOM, ZWSP) al comparar o tokenizar: aparecen «pegados» a la primera palabra de un archivo.
* **Fiarse de la segmentación de grafemas de un runtime antiguo** con emojis recientes.

## 16. Autoevaluación

**1.** **(Cálculo)** ¿Cuántos code points, grafemas y bytes UTF-8 tiene «Ñandú 😀»? (supón que todo está en NFC).

<details>
<summary>Ver respuesta</summary>

Code points: `Ñ a n d ú ␣ 😀` = **7**. Grafemas: **7**. Bytes: Ñ(2)+a(1)+n(1)+d(1)+ú(2)+espacio(1)+😀(4) = **12**.

</details>

**2.** **(Cálculo)** Codifica a mano `U+00F1` (ñ) en UTF-8 y comprueba el resultado.

<details>
<summary>Ver respuesta</summary>

`U+00F1` = `0000 0000 1111 0001`; cae en el rango de 2 bytes (11 bits útiles: `00011 110001`). Prefijos: `110` + `00011` = `11000011` = `C3`; `10` + `110001` = `10110001` = `B1`. Resultado **`C3 B1`**. Lo confirma `'ñ'.encode().hex()`.

</details>

**3.** **(Diagnóstico)** Tu buscador no encuentra «canción» aunque el documento la contiene, y `"canción" == doc_palabra` da `False`. ¿Qué compruebas primero y cómo lo arreglas?

<details>
<summary>Ver respuesta</summary>

Compara las secuencias de code points (`[hex(ord(c)) for c in ...]`): lo más probable es que una esté en NFD (`o`+`U+0301`) y la otra en NFC (`U+00F3`), o que haya un invisible (`U+200B`). Se arregla normalizando a NFC —y quitando invisibles— en ingesta y consulta con la **misma** función.

</details>

**4.** **(Diagnóstico)** Un usuario pega «pаypal.com» (con «а» cirílica) en tu campo de búsqueda y tu filtro de listas de bloqueo no lo detecta. ¿Por qué y qué defensa añadirías?

<details>
<summary>Ver respuesta</summary>

El filtro compara cadenas/bytes y la «а» cirílica son otros bytes (`D0 B0`). Defensas: detectar palabras con escrituras mezcladas (LATIN+CYRILLIC), mapear a un *esqueleto* de confusables (UTS #39) antes de comparar y, en dominios, usar Punycode/IDN con política de escrituras.

</details>

**5.** **(Transferencia)** Vas a entrenar un tokenizer BPE con 50 GB de texto web. ¿Qué decisiones de normalización tomas antes y por qué afectan al vocabulario?

<details>
<summary>Ver respuesta</summary>

Fijar NFC (o NFKC si el uso es de *matching*) para evitar que `é` y `e`+◌́ generen vocabularios duplicados; decidir qué hacer con invisibles (BOM, ZWSP) y controles; decidir si conservar ZWJ/ZWNJ (necesarios en emojis y en persa/indio); y registrar la forma usada, porque en inferencia hay que aplicar **la misma** normalización o los tokens no coincidirán.

</details>

**6.** **(Transferencia)** Diseña una regla para recortar un texto a «no más de 280 caracteres visibles» en una app multiplataforma (web JS, Android Kotlin, backend Python).

<details>
<summary>Ver respuesta</summary>

Medir y cortar en **grafemas** (UAX #29) con la misma versión de Unicode en todas las plataformas, no en `length` (UTF-16 en JS/Kotlin) ni en bytes. Tests con emojis ZWJ, banderas y marcas combinantes.

</details>

**7.** ¿Por qué UTF-8 prohíbe las codificaciones demasiado largas (*overlongs*)?

<details>
<summary>Ver respuesta</summary>

Porque cada code point debe tener **una única** codificación; si no, un filtro que busque la forma canónica de un byte peligroso (`/`, `\0`) podría saltarse con su forma larga (`C0 AF`).

</details>

> **➡️ Siguiente módulo.** **02 · BPE desde cero.** Ya sabes que un tokenizer byte-level ve **bytes UTF-8**. En el módulo 02 implementas el algoritmo de Byte Pair Encoding —nacido como compresor—, la pre-tokenización con regex de GPT-2/cl100k/o200k, el entrenamiento eficiente y la verificación contra `tiktoken`: la normalización que acabas de dominar determina exactamente sobre qué se aprenden las fusiones.

## 17. Referencias
Referencias de arXiv comprobadas (título y autores) el 25-oct-2026. Las normas (RFC, UAX, UTS) se citan por su identificador oficial; no pudieron abrirse desde el entorno de autoría (acceso bloqueado), compruébalas en su sitio oficial.

* Unicode Consortium. *Unicode Standard Annex #15: Unicode Normalization Forms* (UAX #15). https://unicode.org/reports/tr15/
* Unicode Consortium. *Unicode Standard Annex #29: Unicode Text Segmentation* (UAX #29, grafemas). https://unicode.org/reports/tr29/
* Unicode Consortium. *Unicode Technical Standard #39: Unicode Security Mechanisms* (confusables). https://unicode.org/reports/tr39/
* Yergeau, F. (2003). *RFC 3629: UTF-8, a transformation format of ISO 10646.* https://www.rfc-editor.org/rfc/rfc3629
* Pike, R., Thompson, K. (1993). *Hello World or Καλημέρα κόσμε or こんにちは 世界.* USENIX Winter 1993 Technical Conference (historia de UTF-8).
* Boucher, N., Anderson, R. (2021). *Trojan Source: Invisible Vulnerabilities.* arXiv:2111.00169. https://arxiv.org/abs/2111.00169
* Wang, C., Cho, K., Gu, J. (2019). *Neural Machine Translation with Byte-Level Subwords.* arXiv:1909.03341. https://arxiv.org/abs/1909.03341
* Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* arXiv:2305.15425. https://arxiv.org/abs/2305.15425
* Ahia, O. et al. (2023). *Do All Languages Cost the Same? Tokenization in the Era of Commercial Language Models.* arXiv:2305.13707. https://arxiv.org/abs/2305.13707
* Xue, L. et al. (2021). *ByT5: Towards a token-free future with pre-trained byte-to-byte models.* arXiv:2105.13626. https://arxiv.org/abs/2105.13626
* Pagnoni, A. et al. (2024). *Byte Latent Transformer: Patches Scale Better Than Tokens.* arXiv:2412.09871. https://arxiv.org/abs/2412.09871
* Mielke, S. J. et al. (2021). *Between words and characters: A Brief History of Open-Vocabulary Modeling and Tokenization in NLP.* arXiv:2112.10508. https://arxiv.org/abs/2112.10508
* Shannon, C. E. (1948). *A Mathematical Theory of Communication.* Bell System Technical Journal.
* SentencePiece, documentación de normalización: https://github.com/google/sentencepiece/blob/master/doc/normalization.md
* The Register (28-ago-2024). *Microsoft Copilot «Copirate»* (ASCII smuggling, J. Rehberger). https://www.theregister.com/2024/08/28/microsoft_copilot_copirate
* Python: `unicodedata`, `codecs`, PEP 393 (representación flexible de `str`) y PEP 3131 (identificadores no ASCII). · `ftfy` (https://pypi.org/project/ftfy/).